In [ ]:
# CELL 1: Install libraries

!pip install anthropic pandas -q

In [ ]:
# CELL 2: Imports and settings

import pandas as pd
import anthropic
import time
import os
import shutil
import getpass  # lets API key be pasted without it showing on screen
from google.colab import files, drive

# API KEY
# getpass.getpass() pops up a hidden input box at runtime, 
# key should be pasted there, press enter, and it is NEVER written into the
# notebook file itself. 
# Do not paste key into any other cell (code, markdown, or
# output) 

ANTHROPIC_API_KEY = getpass.getpass("Paste Anthropic API key (input hidden): ")

# FILE NAMES
# Renamed from an older CMV project's file names. Keeping the old names
# here would make Cell 5 load the finished CMV file instead of
# this new HH-RLHF data, because Cell 5 only checks "does a file with
# this exact name already exist."

CHECKPOINT_FILE = "hh_subset0_decontext_checkpoint.csv"
OUTPUT_FILE     = "hh_subset0_decontextualised.csv"

# GOOGLE DRIVE PATH
# Points at the HH-RLHF project folder rather than the old CMV
# dissertation_checkpoints folder, so the two pipelines don't share a
# directory and can't overwrite each other's files.

DRIVE_FOLDER     = "/content/drive/MyDrive/HH_RLHF_pipeline/checkpoints"
DRIVE_CHECKPOINT = os.path.join(DRIVE_FOLDER, CHECKPOINT_FILE)

# RUN SETTINGS
# Worth knowing: 144,238 rows at 0.3s sleep alone is roughly 12 hours
# of pure waiting time, before any real API latency is added, treat
# that as an estimate. The checkpointing logic below is built to
# handle that (it resumes where it left off).

CHECKPOINT_EVERY    = 200
SLEEP_BETWEEN_CALLS = 0.3

# API key input
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print("Config loaded.")
print(f"Checkpoint (local) : {CHECKPOINT_FILE}")
print(f"Checkpoint (Drive) : {DRIVE_CHECKPOINT}")

Paste Anthropic API key (input hidden): ··········
Config loaded.
Checkpoint (local) : hh_subset0_decontext_checkpoint.csv
Checkpoint (Drive) : /content/drive/MyDrive/HH_RLHF_pipeline/checkpoints/hh_subset0_decontext_checkpoint.csv


In [ ]:
# CELL 3: Mount Google Drive
# This lets the notebook read/write files in your Drive, so progress
# checkpoints survive even if the Colab session disconnects.

drive.mount("/content/drive")

os.makedirs(DRIVE_FOLDER, exist_ok=True)

print(f"Drive mounted.")
print(f"Checkpoint folder ready: {DRIVE_FOLDER}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted.
Checkpoint folder ready: /content/drive/MyDrive/HH_RLHF_pipeline/checkpoints


In [ ]:
# CELL 4: The instruction sent to the AI model, and the function that sends it
# Each extracted "rule" is domain-specific (e.g. about child support, or
# medical evidence). This step rewrites it as a general principle with
# the specific topic stripped out, so rules from very different
# domains can be compared on equal footing later.

DECONTEXT_PROMPT = """You will be given a rule or norm extracted from a piece of text. The rule may be expressed in domain-specific language tied to a particular topic (e.g. child support, medical evidence, gun control).

Your task is to rewrite the rule as a general prescriptive principle that:

- Preserves the core behavioural or epistemic norm
- Removes references to specific domains, topics, or named situations
- Remains concrete enough to be actionable (do not reduce it to a platitude)
- Is expressed as a clear prescriptive statement (what one should do or how one should reason)
- Do not add information, implications or instructions that are not present in the original rule

Examples:

Input: "When evaluating gun control policies, consider both the immediate safety benefits and the long-term effects on civil liberties."
Output: "When evaluating any policy, weigh both its immediate benefits and its long-term effects on other values."

Input: "In medical debates, do not dismiss anecdotal evidence entirely — it can point to patterns worth investigating."
Output: "Do not dismiss anecdotal evidence entirely, as it can point to patterns worth investigating systematically."

Input: "When someone claims superiority in a domain, ask them to specify measurable evidence rather than accepting the claim at face value."
Output: "Require measurable evidence before accepting claims of superiority or expertise."

Return only the rewritten rule. No explanation, no preamble."""


def decontextualise(rule: str, max_retries: int = 4) -> str:
    """
    Send one rule to Claude Haiku and return the decontextualised version.

    Includes exponential backoff retry logic for transient server errors
    (InternalServerError, RateLimitError etc.) -- these are common during
    long runs and are not bugs, just the server hiccuping.

    Wait times between retries: 5s, 10s, 20s, 40s (doubles each time).
    After 4 failed attempts, returns None so the calling loop can log it
    and move on -- the null will be caught and retried on the next resume.
    """
    # Unchanged logic and model string from your original -- this exact code
    # already ran successfully on the CMV corpus, and "claude-haiku-4-5-20251001"
    # matches the current Haiku 4.5 model identifier, so no change needed here.
    wait = 5

    for attempt in range(max_retries):
        try:
            message = client.messages.create(
                model="claude-haiku-4-5-20251001",
                max_tokens=256,
                messages=[
                    {
                        "role": "user",
                        "content": f"{DECONTEXT_PROMPT}\n\nInput: {rule}"
                    }
                ]
            )
            return message.content[0].text.strip()

        except Exception as e:
            error_name = type(e).__name__

            if attempt < max_retries - 1:
                print(f"    [{error_name}] attempt {attempt + 1} -- "
                      f"waiting {wait}s before retry...")
                time.sleep(wait)
                wait *= 2
            else:
                print(f"    [{error_name}] -- all {max_retries} attempts failed. "
                      f"Rule left as null, will retry on next resume.")
                return None

    return None


print("Prompt and function loaded (with retry logic).")

Prompt and function loaded (with retry logic).


In [ ]:
# CELL 5: Load existing progress, or start fresh
# If a checkpoint already exists in Drive (from a previous session),
# load it directly and carry on. Otherwise, this is a first run, so
# upload the raw extraction file (hh_subset_0_rules.csv).

if os.path.exists(DRIVE_CHECKPOINT):
    print("Checkpoint found in Drive -- loading directly, no upload needed.")
    print(f"  Path: {DRIVE_CHECKPOINT}")

    shutil.copy(DRIVE_CHECKPOINT, CHECKPOINT_FILE)
    df = pd.read_csv(CHECKPOINT_FILE)

else:
    print("No checkpoint found in Drive.")
    print("Upload your subset 0 extraction file (hh_subset_0_rules.csv):")
    uploaded = files.upload()

    # Reads whatever filename was actually uploaded, rather than
    # assuming it's already named CHECKPOINT_FILE.
    uploaded_filename = list(uploaded.keys())[0]
    print(f"  Uploaded: {uploaded_filename} "
          f"({os.path.getsize(uploaded_filename):,} bytes)")

    df = pd.read_csv(uploaded_filename)

    # Fresh extraction file has no rule_decontextualised column yet
    # add it now so the "what's left to process" logic below works.
    if "rule_decontextualised" not in df.columns:
        df["rule_decontextualised"] = None
        print("  Added empty 'rule_decontextualised' column (first run).")

    # From here on, save under the checkpoint name and back up to Drive
    df.to_csv(CHECKPOINT_FILE, index=False)
    shutil.copy(CHECKPOINT_FILE, DRIVE_CHECKPOINT)
    print(f"Checkpoint backed up to Drive: {DRIVE_CHECKPOINT}")

# Defensive check, in case a Drive checkpoint somehow exists without this column
# cheap to guard against even if it shouldn't happen.
if "rule_decontextualised" not in df.columns:
    df["rule_decontextualised"] = None

# SHOW CURRENT PROGRESS
done  = df["rule_decontextualised"].notna().sum()
left  = df["rule_decontextualised"].isna().sum()
total = len(df)

print(f"\nTotal rules  : {total:,}")
print(f"Already done : {done:,}")
print(f"Remaining    : {left:,}")
print(f"\nReady to resume. Run Cell 6.")

Checkpoint found in Drive -- loading directly, no upload needed.
  Path: /content/drive/MyDrive/HH_RLHF_pipeline/checkpoints/hh_subset0_decontext_checkpoint.csv

Total rules  : 144,238
Already done : 46,800
Remaining    : 97,438

Ready to resume. Run Cell 6.


/tmp/ipykernel_107482/1026328459.py:10: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CHECKPOINT_FILE)


In [ ]:
# CELL 6: The main processing loop, with checkpointing
# Goes through every rule that has not been decontextualised yet, sends
# it to the AI model, and saves progress to Drive every 200 rows, so
# if the session disconnects (likely, given the ~12 hour floor), you
# can just re-run this cell and it picks up where it left off.

to_process_idx = df[df["rule_decontextualised"].isna()].index

print(f"Rules remaining : {len(to_process_idx):,}")
print(f"Already done    : {df['rule_decontextualised'].notna().sum():,}")
print(f"Total           : {len(df):,}")
print()

if len(to_process_idx) == 0:
    print("All rules already processed. Run Cell 7 to check quality.")

else:
    error_count = 0

    for i, idx in enumerate(to_process_idx):

        rule = df.at[idx, "rule"]

        try:
            df.at[idx, "rule_decontextualised"] = decontextualise(rule)

        except Exception as e:
            error_count += 1
            print(f"  ERROR at index {idx}: {type(e).__name__}: {e}")
            df.at[idx, "rule_decontextualised"] = None
            time.sleep(2)

        time.sleep(SLEEP_BETWEEN_CALLS)

        if (i + 1) % CHECKPOINT_EVERY == 0:
            df.to_csv(CHECKPOINT_FILE, index=False)
            shutil.copy(CHECKPOINT_FILE, DRIVE_CHECKPOINT)

            done_so_far = df["rule_decontextualised"].notna().sum()
            print(f"  Checkpoint saved | "
                  f"{i+1:,}/{len(to_process_idx):,} this session | "
                  f"{done_so_far:,} total done | "
                  f"{error_count} errors")

    df.to_csv(OUTPUT_FILE, index=False)
    shutil.copy(OUTPUT_FILE, os.path.join(DRIVE_FOLDER, OUTPUT_FILE))

    nulls_remaining = df["rule_decontextualised"].isna().sum()

    print()
    print("=" * 70)
    print("Run complete.")
    print(f"  Rules processed : {df['rule_decontextualised'].notna().sum():,}")
    print(f"  Nulls remaining : {nulls_remaining}")
    print(f"  Total errors    : {error_count}")
    print(f"  Saved locally   : {OUTPUT_FILE}")
    print(f"  Saved to Drive  : {os.path.join(DRIVE_FOLDER, OUTPUT_FILE)}")

    if nulls_remaining > 0:
        print(f"\n  {nulls_remaining} rules failed -- re-run this cell to retry them.")
        print(f"  The checkpoint will skip everything already completed.")

Rules remaining : 97,438
Already done    : 46,800
Total           : 144,238

  Checkpoint saved | 200/97,438 this session | 47,000 total done | 0 errors
  Checkpoint saved | 400/97,438 this session | 47,200 total done | 0 errors
  Checkpoint saved | 600/97,438 this session | 47,400 total done | 0 errors
  Checkpoint saved | 800/97,438 this session | 47,600 total done | 0 errors
  Checkpoint saved | 1,000/97,438 this session | 47,800 total done | 0 errors


In [ ]:
# CELL 7: Quality-check the finished output
# Checks for two known failure modes in the AI's rewrites:
#   1. Meaning expansion: the model added implications that weren't
#      in the original rule
#   2. Meaning inversion: the model reversed the rule's direction
#      (e.g. turned a "should" into a "should not")

result_df = pd.read_csv(OUTPUT_FILE)

print("=== COMPLETION STATS ===")
print(f"Total rules      : {len(result_df):,}")
print(f"Decontextualised : {result_df['rule_decontextualised'].notna().sum():,}")
print(f"Still null       : {result_df['rule_decontextualised'].isna().sum():,}")

print("\n=== RANDOM SAMPLE (20 rules) ===")
print("Read these carefully -- flag any inversions or expansions")
print("-" * 70)

sample = result_df[result_df["rule_decontextualised"].notna()].sample(20, random_state=99)

for _, row in sample.iterrows():
    # CHANGED: 'topic_label' -> 'topic'. Your subset 0 file has a bare integer
    # topic id column (confirmed by direct inspection), not a topic_label
    # string column -- the original line would have raised a KeyError here.
    print(f"\n  topic    : {row['topic']}")
    print(f"  ORIGINAL : {row['rule']}")
    print(f"  OUTPUT   : {row['rule_decontextualised']}")

print("\n=== WORD COUNT CHECK ===")
result_df["decontext_word_count"] = result_df["rule_decontextualised"].str.split().str.len()
long_outputs = result_df[result_df["decontext_word_count"] > 40]
print(f"Outputs over 40 words: {len(long_outputs):,}")

if len(long_outputs) > 0:
    print("Sample of long outputs:")
    for _, row in long_outputs.sample(min(5, len(long_outputs)), random_state=42).iterrows():
        print(f"  [{row['decontext_word_count']} words] {row['rule_decontextualised']}")

In [ ]:
# CELL 8: Download the finished file

files.download(OUTPUT_FILE)
print(f"Downloaded: {OUTPUT_FILE}")